<a href="https://colab.research.google.com/github/SattikBhaumik/CS156/blob/main/CS156_Session_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Gradients: Multivariate Derivatives

##Sigmoid example

In [1]:
import numpy as np

# Forward pass
w0, w1, w2 = 2.00, -3.00, -3.00
x0, x1 = -1.00, -2.00

mul0 = w0 * x0  # -2.00
mul1 =  w1 * x1  # 6.00
sum01 = mul0 + mul1  # 4.00
sum012 = sum01 + w2  # 1.00

neg = -sum012  # -1.00
exp = np.exp(neg)  # 0.37
plus1 = exp + 1  # 1.37
invert = 1/plus1  # 0.73


# Backward pass
# Remember *all* derivatives are with respect to the final function invert
d_invert = 1.00  # this is d_invertd_invert, but we shorten to d_invert
# Here's one completed for you :)
d_plus1 = d_invert * (-1/plus1**2)  # this is d_invertd_plus_1, but we shorten to d_plus_1
d_exp = d_plus1 * 1.00
d_neg = d_exp * exp
d_sum012 = d_neg * (-1)

# Notice for gates with multple inputs, when going backward
# they are responsible for derivatives of multiple values!
d_w2, d_sum01 = d_sum012, d_sum012
d_mul0, d_mul1 = d_sum01, d_sum01

d_w1, d_x1 = d_mul1*x1, d_mul1*w1
d_w0, d_x0 = d_mul0*x0, d_mul0*w0

In [2]:
# Forward pass
w = np.array([2.00, -3.00])
b = -3.00

x = np.array([-1.00, -2.00])

dot = w.T @ x  # 4.00
summed = dot + b  # 1.00

neg = -summed  # -1.00
exp = np.exp(neg)  # 0.37
plus1 = exp + 1  # 1.37
invert = 1/plus1  # 0.73


# Backward pass
d_invert = 1.00
d_plus1 = d_invert * (-1/ plus1**2)
d_exp = d_plus1 * 1
d_neg = d_neg * exp
summed = d_neg * (-1)

d_b, d_dot = summed, summed
# Be careful, the next two are vectors!
d_w = d_dot * x
d_x = d_b * w

##Sigmoid example (vectorized)

In [3]:
# Forward pass
w = np.array([2.00, -3.00])
b = -3.00

x = np.array([-1.00, -2.00])

dot = w.T @ x  # 4.00
summed = dot + b  # 1.00

neg = -summed  # -1.00
exp = np.exp(neg)  # 0.37
plus1 = exp + 1  # 1.37
invert = 1/plus1  # 0.73


# Backward pass
d_invert = 1.00
d_plus1 = d_invert * (-1/ plus1**2)
d_exp = d_plus1 * 1
d_neg = d_neg * exp
summed = d_neg * (-1)

d_b, d_dot = summed, summed
# Be careful, the next two are vectors!
d_w = d_dot * x
d_x = d_b * w

##Auto-magic gradients

In [4]:
%%capture
!python -m pip install -U --no-cache-dir "jax==0.4.13" "jaxlib==0.4.13" -f https://storage.googleapis.com/jax-releases/jax_releases.html

In [5]:
import jax
import jax.numpy as jnp  # JAX has a version of NumPy that acts...just like NumPy!

# We first define a function for *just* the forward pass
def sigmoid_example(params, x):
    dot = params['w'].T @ x  # 4.00
    summed = dot + params['b']  # 1.00

    neg = -summed  # -1.00
    exp = jnp.exp(neg)  # 0.37
    plus1 = exp + 1  # 1.37
    invert = 1/plus1  # 0.73

    return invert

In [6]:
params = {'w': np.array([2.00, -3.00]), 'b': -3.00}  # We define our parameters as a dict.
x = np.array([-1.00, -2.00])

sigmoid_example(params, x)  # We get the expected value of 0.73 back.

Array(0.7310586, dtype=float32)

In [7]:
grad_sigmoid_example = jax.grad(sigmoid_example)
grad_sigmoid_example(params, x)

{'b': Array(0.19661197, dtype=float32, weak_type=True),
 'w': Array([-0.19661197, -0.39322394], dtype=float32)}

##training

In [8]:
# Best wishes!
'''
I have decided to put both my code and AI assisted code because I wanted to see the differences. AI went with using a tree map kind of function from jax library whereas I just used the normal numpy approach but with jnp
'''

# copying code from code cell 4, 5 and 6
# I checked my code with claude and chatgpt because i needed help with jax library and documentation.
# following code is mine and comments as well.
# I did use AI to correct for one bug that I was having with iteration and learning rate issue, but that is solved.

# My Code answer
# We first define a function for *just* the forward pass
def sigmoid_example(params, x):
    dot = params['w'].T @ x  # 4.00
    summed = dot + params['b']  # 1.00
    neg = -summed  # -1.00
    exp = jnp.exp(neg)  # 0.37
    plus1 = exp + 1  # 1.37
    invert = 1/plus1  # 0.73
    return invert

# jax.grad differentiates with respect to the first argument (params).
# we get a dictionary as output with the params structure:
# {'w': dy/dw, 'b': dy/db}
grad_sigmoid_example = jax.grad(sigmoid_example)

# this function calculates the gradient descent
def new_function(params, x, learning_rate=0.1, iterations=1000):
    for iteration in range(iterations):
        grads = grad_sigmoid_example(params, x)
        # We define our parameters as a dictionary
        params = {'w': params['w'] - learning_rate * grads['w'],
                  'b': params['b'] - learning_rate * grads['b']}
    return params

# initial parameters with an input vector x
params = {'w': np.array([2.00, -3.00]), 'b': -3.00}
x = np.array([-1.00, -2.00])

# 1000 iterations (calculating gradient 1000 times)
trained_params = new_function(params,x,learning_rate=0.1,iterations=1000)
print("Results from My Code:")
print("Initial output:", sigmoid_example(params, x))
print("Trained parameters:", trained_params)
print("Final output:", sigmoid_example(trained_params, x))
print()


# AI Assisted Code
import jax
import jax.numpy as jnp

# here, the forward pass happens in two lines including the logistic transformation
def sigmoid_example(params, x):
    """Forward pass: sigma(z) = 1 / (1 + exp(-z)), where z = w.x + b."""
    # linear pre-activation (scalar)
    z = params['w'] @ x + params['b']
    # logistic transformation of z into (0, 1)
    return 1 / (1 + jnp.exp(-z))

## following comment is from chatgpt which i have included because i found it helpful
# jax.grad differentiates w.r.t. the first argument: params,
# so the result output is a pytree with the same structure as params: {'w': dy/dw, 'b': dy/db}
grad_function = jax.grad(sigmoid_example)

def train(params, x, lr=0.1, iterations=1000):
    """Gradient descent:
    theta <- theta - lr * dy/dtheta,
    applied to every leaf."""
    for _ in range(iterations):
        grads = grad_function(params, x)
        # this code line was suggested by claude which i applied here
        params = jax.tree_util.tree_map(lambda p, g: p - lr * g, params, grads)
    return params

params = {'w': jnp.array([2.0, -3.0]), 'b': -3.0}
x = jnp.array([-1.0, -2.0])
trained = train(params, x)
# z = 1 -> ~0.73
print("Results from AI Assisted Code:")
print("Initial output:", sigmoid_example(params, x))
print("Trained parameters:", trained)
print("Final output:", sigmoid_example(trained, x))

Results from My Code:
Initial output: 0.7310586
Trained parameters: {'w': Array([ 3.2284188 , -0.54317373], dtype=float32), 'b': Array(-4.22841, dtype=float32, weak_type=True)}
Final output: 0.0017084121

Results from AI Assisted Code:
Initial output: 0.7310586
Trained parameters: {'b': Array(-4.22841, dtype=float32, weak_type=True), 'w': Array([ 3.2284188 , -0.54317373], dtype=float32)}
Final output: 0.0017084121
